In [3]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from dotenv import load_dotenv
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torchvision import models, transforms

from dataset import BreedDataset

Reproducibility

In [16]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

DEVICE = 'cuda'

Load Dataset Metadata

In [4]:
load_dotenv()
DATASET = os.getenv('DATASET_PATH')

mat_data = loadmat(f'{DATASET}/file_list.mat')

data = []

for num, img_path in enumerate(mat_data['file_list']):
    data.append({
        'img_path': f'{DATASET}/images/{str(img_path[0][0])}',
        'annotation_path': f'{DATASET}/annotation/{mat_data['annotation_list'][num][0][0]}',
        'label': mat_data['labels'][num][0],
    })

df = pd.DataFrame(data)

df.head()

,img_path,annotation_path,label
0,/home/danylo/GIT/dog-classifier/dataset/images...,/home/danylo/GIT/dog-classifier/dataset/annota...,1
1,/home/danylo/GIT/dog-classifier/dataset/images...,/home/danylo/GIT/dog-classifier/dataset/annota...,1
2,/home/danylo/GIT/dog-classifier/dataset/images...,/home/danylo/GIT/dog-classifier/dataset/annota...,1
3,/home/danylo/GIT/dog-classifier/dataset/images...,/home/danylo/GIT/dog-classifier/dataset/annota...,1
4,/home/danylo/GIT/dog-classifier/dataset/images...,/home/danylo/GIT/dog-classifier/dataset/annota...,1


Data Augmentations

In [9]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    ),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    ),
])

Datasets and DataLoaders

In [10]:
df = pd.DataFrame(data)

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

train_dataset = BreedDataset(
    train_df,
    transform=train_transform
)

val_dataset = BreedDataset(
    val_df,
    transform=val_transform
)

test_dataset = BreedDataset(
    test_df,
    transform=val_transform
)

g = torch.Generator()
g.manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    generator=g,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    generator=g,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    generator=g,
)

Get Model

In [11]:
def get_model(num_classes=120):
    mdl = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

    for param in mdl.parameters():
        param.requires_grad = False

    for param in mdl.layer3.parameters():
        param.requires_grad = True

    for param in mdl.layer4.parameters():
        param.requires_grad = True

    mdl.fc = nn.Linear(mdl.fc.in_features, num_classes) # because 120 classes

    return mdl.to('cuda')

Train Epoch

In [12]:
def train_epoch(
    mdl,
    loader,
    criterion,
    optimizer,
):
    mdl.train()

    total_loss = 0
    correct = 0

    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = mdl(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (
            outputs.argmax(1) == labels
        ).sum().item()

    return (
        total_loss / len(loader),
        correct / len(loader.dataset),
    )

In [13]:
@torch.no_grad()
def eval_epoch(
    mdl,
    loader,
    criterion,
):
    mdl.eval()

    total_loss = 0
    correct = 0

    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = mdl(imgs)
        loss = criterion(outputs, labels)

        total_loss += loss.item()
        correct += (
            outputs.argmax(1) == labels
        ).sum().item()

    return (
        total_loss / len(loader),
        correct / len(loader.dataset),
    )

Training Loop

In [14]:
def train(
    mdl,
    t_loader,
    v_loader,
    epochs=10,
):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(
        [
            {'params': mdl.layer3.parameters(), 'lr': 1e-5},
            {'params': mdl.layer4.parameters(), 'lr': 3e-5},
            {'params': mdl.fc.parameters(), 'lr': 1e-4},
        ],
        weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    best_val_acc = 0

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(mdl, t_loader, criterion, optimizer)
        val_loss, val_acc = eval_epoch(mdl, v_loader, criterion)
        scheduler.step()

        print(
            f'Epoch {epoch + 1}/{epochs}'
            f' | train loss: {train_loss:.4f}'
            f' acc: {train_acc:.4f}'
            f' | val loss: {val_loss:.4f}'
            f' acc: {val_acc:.4f}'
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(mdl.state_dict(), 'best_model.pth')
            print(f'  -> saved best model (val_acc={val_acc:.4f})')


Start Training

In [17]:
model = get_model(num_classes=120)

train(model, train_loader, val_loader, epochs=10)

Epoch 1/10 | train loss: 2.8106 acc: 0.5490 | val loss: 1.4453 acc: 0.8163
  -> saved best model (val_acc=0.8163)
Epoch 2/10 | train loss: 1.3796 acc: 0.8316 | val loss: 1.2547 acc: 0.8520
  -> saved best model (val_acc=0.8520)
Epoch 3/10 | train loss: 1.2232 acc: 0.8712 | val loss: 1.2099 acc: 0.8659
  -> saved best model (val_acc=0.8659)
Epoch 4/10 | train loss: 1.1332 acc: 0.9031 | val loss: 1.1914 acc: 0.8682
  -> saved best model (val_acc=0.8682)
Epoch 5/10 | train loss: 1.1174 acc: 0.9106 | val loss: 1.1880 acc: 0.8740
  -> saved best model (val_acc=0.8740)
Epoch 6/10 | train loss: 1.1017 acc: 0.9139 | val loss: 1.1843 acc: 0.8691
Epoch 7/10 | train loss: 1.0969 acc: 0.9170 | val loss: 1.1834 acc: 0.8707
Epoch 8/10 | train loss: 1.0966 acc: 0.9152 | val loss: 1.1810 acc: 0.8695
Epoch 9/10 | train loss: 1.0961 acc: 0.9168 | val loss: 1.1819 acc: 0.8711
Epoch 10/10 | train loss: 1.0939 acc: 0.9164 | val loss: 1.1784 acc: 0.8759
  -> saved best model (val_acc=0.8759)
